# Basic Chain Ladder IBNR: `chainladder` package

Same data and set-up as the pandas notebook (origin = month of loss, development = reporting lag in months, valued at 31 Dec 2025), this time using the [`chainladder`](https://chainladder-python.readthedocs.io) package. Results are reconciled against the manual calculation at the end.

In [2]:
import pandas as pd
import numpy as np
import chainladder as cl

pd.options.display.float_format = "{:,.4f}".format

In [ ]:

PATH = "data__2_.xlsx"
VALUATION_DATE = pd.Timestamp("2025-12-31")


def parse_date(value):
    if isinstance(value, str):
        return pd.to_datetime(value.strip(), format="%d.%m.%Y", errors="coerce")
    return pd.to_datetime(value, errors="coerce")


def load_claims(path=PATH, valuation_date=VALUATION_DATE):
    raw = pd.read_excel(path)
    claims = raw.iloc[:, [3, 9, 13, 18]].copy()
    claims.columns = ["claim_no", "loss_date", "report_date", "incurred"]
    claims = claims.dropna(subset=["claim_no"]).drop_duplicates()
    claims["loss_date"] = claims["loss_date"].map(parse_date)
    claims["report_date"] = claims["report_date"].map(parse_date)
    claims["incurred"] = pd.to_numeric(claims["incurred"], errors="coerce")
    valid = (
        claims["loss_date"].between("2025-01-01", valuation_date)
        & claims["report_date"].between(claims["loss_date"], valuation_date)
        & claims["incurred"].notna()
    )
    return claims[valid].reset_index(drop=True)


## 1. Data

In [2]:
claims = load_claims().assign(count=1)
claims.head()

/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:223: UserWarning: Cell I645 is marked as a date but the serial value 6062025 is outside the limits for dates. The cell will be treated as an error.
  warn(msg)


,claim_no,loss_date,report_date,incurred,count
0,010/401/1/080651/1015,2025-09-30,2025-10-03,"24,570.0000",1
1,010/401/9/000023/2025/11,2025-11-19,2025-11-26,"22,164.4000",1
2,010/401/9/000024/2025/11,2025-11-10,2025-12-03,"46,263.0000",1
3,010/401/9/000026/2025/12,2025-11-20,2025-12-01,"24,400.0000",1
4,010/401/9/000028/2025/12,2025-11-20,2025-12-09,"83,880.0000",1


## 2. Triangles

In [3]:
def make_triangle(claims, column):
    tri = cl.Triangle(
        claims,
        origin="loss_date",
        development="report_date",
        columns=[column],
        cumulative=False,
    )
    return tri.incr_to_cum()


count_tri = make_triangle(claims, "count")
amount_tri = make_triangle(claims, "incurred")
print(count_tri)
count_tri.to_frame(origin_as_datetime=False)

             1       2       3       4       5       6       7       8       9       10      11      12
2025-01 19.0000 31.0000 35.0000 35.0000 36.0000 36.0000 36.0000 37.0000 38.0000 38.0000 38.0000 38.0000
2025-02 16.0000 27.0000 29.0000 29.0000 30.0000 30.0000 30.0000 30.0000 30.0000 30.0000 30.0000     NaN
2025-03 28.0000 44.0000 48.0000 50.0000 51.0000 51.0000 51.0000 51.0000 52.0000 52.0000     NaN     NaN
2025-04 21.0000 39.0000 45.0000 47.0000 47.0000 47.0000 47.0000 47.0000 47.0000     NaN     NaN     NaN
2025-05 22.0000 37.0000 44.0000 46.0000 47.0000 48.0000 48.0000 48.0000     NaN     NaN     NaN     NaN
2025-06 14.0000 39.0000 45.0000 50.0000 51.0000 53.0000 53.0000     NaN     NaN     NaN     NaN     NaN
2025-07 16.0000 37.0000 46.0000 48.0000 49.0000 50.0000     NaN     NaN     NaN     NaN     NaN     NaN
2025-08 21.0000 46.0000 54.0000 54.0000 55.0000     NaN     NaN     NaN     NaN     NaN     NaN     NaN
2025-09 21.0000 42.0000 44.0000 45.0000     NaN     NaN     NaN 

,1,2,3,4,5,6,7,8,9,10,11,12
2025-01,19.0000,31.0000,35.0000,35.0000,36.0000,36.0000,36.0000,37.0000,38.0000,38.0000,38.0000,38.0000
2025-02,16.0000,27.0000,29.0000,29.0000,30.0000,30.0000,30.0000,30.0000,30.0000,30.0000,30.0000,NaN
2025-03,28.0000,44.0000,48.0000,50.0000,51.0000,51.0000,51.0000,51.0000,52.0000,52.0000,NaN,NaN
2025-04,21.0000,39.0000,45.0000,47.0000,47.0000,47.0000,47.0000,47.0000,47.0000,NaN,NaN,NaN
2025-05,22.0000,37.0000,44.0000,46.0000,47.0000,48.0000,48.0000,48.0000,NaN,NaN,NaN,NaN
2025-06,14.0000,39.0000,45.0000,50.0000,51.0000,53.0000,53.0000,NaN,NaN,NaN,NaN,NaN
2025-07,16.0000,37.0000,46.0000,48.0000,49.0000,50.0000,NaN,NaN,NaN,NaN,NaN,NaN
2025-08,21.0000,46.0000,54.0000,54.0000,55.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-09,21.0000,42.0000,44.0000,45.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-10,24.0000,41.0000,49.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
amount_tri.to_frame(origin_as_datetime=False).round(0)

,1,2,3,4,5,6,7,8,9,10,11,12
2025-01,"323,150.0000","574,467.0000","819,682.0000","819,682.0000","846,682.0000","846,682.0000","846,682.0000","856,132.0000","860,632.0000","860,632.0000","860,632.0000","860,632.0000"
2025-02,"532,751.0000","693,256.0000","792,893.0000","792,893.0000","992,893.0000","992,893.0000","992,893.0000","992,893.0000","992,893.0000","992,893.0000","992,893.0000",NaN
2025-03,"991,784.0000","1,669,283.0000","1,824,579.0000","1,961,276.0000","1,979,906.0000","1,979,906.0000","1,979,906.0000","1,979,906.0000","2,009,906.0000","2,009,906.0000",NaN,NaN
2025-04,"810,046.0000","1,699,684.0000","2,184,635.0000","2,197,098.0000","2,197,098.0000","2,197,098.0000","2,197,098.0000","2,197,098.0000","2,197,098.0000",NaN,NaN,NaN
2025-05,"310,729.0000","813,521.0000","893,050.0000","920,822.0000","942,746.0000","1,031,846.0000","1,031,846.0000","1,031,846.0000",NaN,NaN,NaN,NaN
2025-06,"775,132.0000","1,436,889.0000","1,770,825.0000","1,990,586.0000","1,995,586.0000","2,028,576.0000","2,028,576.0000",NaN,NaN,NaN,NaN,NaN
2025-07,"364,321.0000","1,696,722.0000","2,319,829.0000","2,334,798.0000","2,351,898.0000","2,351,898.0000",NaN,NaN,NaN,NaN,NaN,NaN
2025-08,"678,289.0000","1,670,327.0000","1,958,297.0000","1,958,297.0000","1,985,297.0000",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-09,"481,125.0000","1,286,133.0000","1,330,384.0000","1,348,384.0000",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-10,"671,043.0000","1,304,331.0000","1,526,352.0000",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Development factors

In [5]:
def fit_development(tri, average="volume"):
    return cl.Development(average=average).fit(tri)


dev_count = fit_development(count_tri)
dev_count.ldf_.to_frame(origin_as_datetime=False)

,1-2,2-3,3-4,4-5,5-6,6-7,7-8,8-9,9-10,10-11,11-12
(All),1.9099,1.1462,1.0359,1.0195,1.0129,1.0000,1.0047,1.0121,1.0000,1.0000,1.0000


In [6]:
ldf_compare = pd.concat(
    [fit_development(count_tri, avg).ldf_.to_frame(origin_as_datetime=False).iloc[0].rename(avg) for avg in ["volume", "simple", "regression"]],
    axis=1,
)
ldf_compare

,volume,simple,regression
1-2,1.9099,1.9524,1.8714
2-3,1.1462,1.1451,1.1466
3-4,1.0359,1.0343,1.0367
4-5,1.0195,1.0205,1.0188
5-6,1.0129,1.0116,1.0139
6-7,1.0000,1.0000,1.0000
7-8,1.0047,1.0056,1.0039
8-9,1.0121,1.0117,1.0124
9-10,1.0000,1.0000,1.0000
10-11,1.0000,1.0000,1.0000


## 4. Chain ladder

In [7]:
def run_chainladder(tri, average="volume", tail=1.0):
    pipe = cl.Pipeline([
        ("dev", cl.Development(average=average)),
        ("tail", cl.TailConstant(tail=tail)),
        ("model", cl.Chainladder()),
    ])
    return pipe.fit(tri)


def result_table(model, tri):
    m = model.named_steps["model"]
    latest = tri.latest_diagonal.to_frame(origin_as_datetime=False).iloc[:, 0]
    ult = m.ultimate_.to_frame(origin_as_datetime=False).iloc[:, 0]
    ibnr = m.ibnr_.to_frame(origin_as_datetime=False).iloc[:, 0]
    return pd.DataFrame({"reported": latest, "ultimate": ult, "ibnr": ibnr})


count_model = run_chainladder(count_tri)
count_result = result_table(count_model, count_tri)
count_result

/usr/local/lib/python3.12/dist-packages/chainladder/tails/base.py:133: RuntimeWarning: overflow encountered in exp
  sigma_ = xp.exp(time_pd * reg.slope_ + reg.intercept_)
/usr/local/lib/python3.12/dist-packages/chainladder/tails/base.py:137: RuntimeWarning: overflow encountered in exp
  std_err_ = xp.exp(time_pd * reg.slope_ + reg.intercept_)
/usr/local/lib/python3.12/dist-packages/chainladder/tails/base.py:140: RuntimeWarning: invalid value encountered in multiply
  sigma_ = sigma_ * 0
/usr/local/lib/python3.12/dist-packages/chainladder/tails/base.py:141: RuntimeWarning: invalid value encountered in multiply
  std_err_ = std_err_ * 0


,reported,ultimate,ibnr
2025-01,38.0000,38.0000,NaN
2025-02,30.0000,30.0000,NaN
2025-03,52.0000,52.0000,NaN
2025-04,47.0000,47.0000,NaN
2025-05,48.0000,48.5818,0.5818
2025-06,53.0000,53.8955,0.8955
2025-07,50.0000,50.8448,0.8448
2025-08,55.0000,56.6486,1.6486
2025-09,45.0000,47.2526,2.2526
2025-10,49.0000,53.2998,4.2998


In [8]:
amount_model = run_chainladder(amount_tri)
amount_result = result_table(amount_model, amount_tri)
amount_result.round(2)

/usr/local/lib/python3.12/dist-packages/chainladder/tails/base.py:133: RuntimeWarning: overflow encountered in exp
  sigma_ = xp.exp(time_pd * reg.slope_ + reg.intercept_)
/usr/local/lib/python3.12/dist-packages/chainladder/tails/base.py:137: RuntimeWarning: overflow encountered in exp
  std_err_ = xp.exp(time_pd * reg.slope_ + reg.intercept_)
/usr/local/lib/python3.12/dist-packages/chainladder/tails/base.py:140: RuntimeWarning: invalid value encountered in multiply
  sigma_ = sigma_ * 0
/usr/local/lib/python3.12/dist-packages/chainladder/tails/base.py:141: RuntimeWarning: invalid value encountered in multiply
  std_err_ = std_err_ * 0


,reported,ultimate,ibnr
2025-01,"860,631.5600","860,631.5600",NaN
2025-02,"992,893.1100","992,893.1100",NaN
2025-03,"2,009,906.2400","2,009,906.2400",NaN
2025-04,"2,197,097.7400","2,197,097.7400",NaN
2025-05,"1,031,846.4800","1,037,753.9700","5,907.4900"
2025-06,"2,028,576.0500","2,042,925.3100","14,349.2600"
2025-07,"2,351,898.3100","2,368,534.6100","16,636.3000"
2025-08,"1,985,297.1100","2,020,928.9500","35,631.8400"
2025-09,"1,348,383.9500","1,406,081.2200","57,697.2700"
2025-10,"1,526,352.1500","1,640,885.1900","114,533.0400"


In [9]:
count_model.named_steps['model'].full_triangle_.to_frame(origin_as_datetime=False).round(1)

,1,2,3,4,5,6,7,8,9,10,...,16,17,18,19,20,21,22,23,24,9999
2025-01,19.0000,31.0000,35.0000,35.0000,36.0000,36.0000,36.0000,37.0000,38.0000,38.0000,...,38.0000,38.0000,38.0000,38.0000,38.0000,38.0000,38.0000,38.0000,38.0000,38.0000
2025-02,16.0000,27.0000,29.0000,29.0000,30.0000,30.0000,30.0000,30.0000,30.0000,30.0000,...,30.0000,30.0000,30.0000,30.0000,30.0000,30.0000,30.0000,30.0000,30.0000,30.0000
2025-03,28.0000,44.0000,48.0000,50.0000,51.0000,51.0000,51.0000,51.0000,52.0000,52.0000,...,52.0000,52.0000,52.0000,52.0000,52.0000,52.0000,52.0000,52.0000,52.0000,52.0000
2025-04,21.0000,39.0000,45.0000,47.0000,47.0000,47.0000,47.0000,47.0000,47.0000,47.0000,...,47.0000,47.0000,47.0000,47.0000,47.0000,47.0000,47.0000,47.0000,47.0000,47.0000
2025-05,22.0000,37.0000,44.0000,46.0000,47.0000,48.0000,48.0000,48.0000,48.6000,48.6000,...,48.6000,48.6000,48.6000,48.6000,48.6000,48.6000,48.6000,48.6000,48.6000,48.6000
2025-06,14.0000,39.0000,45.0000,50.0000,51.0000,53.0000,53.0000,53.3000,53.9000,53.9000,...,53.9000,53.9000,53.9000,53.9000,53.9000,53.9000,53.9000,53.9000,53.9000,53.9000
2025-07,16.0000,37.0000,46.0000,48.0000,49.0000,50.0000,50.0000,50.2000,50.8000,50.8000,...,50.8000,50.8000,50.8000,50.8000,50.8000,50.8000,50.8000,50.8000,50.8000,50.8000
2025-08,21.0000,46.0000,54.0000,54.0000,55.0000,55.7000,55.7000,56.0000,56.6000,56.6000,...,56.6000,56.6000,56.6000,56.6000,56.6000,56.6000,56.6000,56.6000,56.6000,56.6000
2025-09,21.0000,42.0000,44.0000,45.0000,45.9000,46.5000,46.5000,46.7000,47.3000,47.3000,...,47.3000,47.3000,47.3000,47.3000,47.3000,47.3000,47.3000,47.3000,47.3000,47.3000
2025-10,24.0000,41.0000,49.0000,50.8000,51.7000,52.4000,52.4000,52.7000,53.3000,53.3000,...,53.3000,53.3000,53.3000,53.3000,53.3000,53.3000,53.3000,53.3000,53.3000,53.3000


## 5. Summary

In [10]:
summary = pd.concat(
    [count_result.sum().rename("claim_count"), amount_result.sum().rename("incurred_amount")], axis=1
)
summary.loc["ibnr_pct_of_reported"] = summary.loc["ibnr"] / summary.loc["reported"] * 100
summary

,claim_count,incurred_amount
reported,527.0000,"18,767,474.6000"
ultimate,573.8858,"20,987,674.0356"
ibnr,46.8858,"2,220,199.4356"
ibnr_pct_of_reported,8.8967,11.8300


## 6. Method comparison

In [11]:
def total_ibnr(tri, average):
    return run_chainladder(tri, average).named_steps["model"].ibnr_.sum().sum()


comparison = pd.DataFrame(
    {avg: [total_ibnr(count_tri, avg), total_ibnr(amount_tri, avg)] for avg in ["volume", "simple", "regression"]},
    index=["claim_count", "incurred_amount"],
)
comparison

/usr/local/lib/python3.12/dist-packages/chainladder/tails/base.py:133: RuntimeWarning: overflow encountered in exp
  sigma_ = xp.exp(time_pd * reg.slope_ + reg.intercept_)
/usr/local/lib/python3.12/dist-packages/chainladder/tails/base.py:137: RuntimeWarning: overflow encountered in exp
  std_err_ = xp.exp(time_pd * reg.slope_ + reg.intercept_)
/usr/local/lib/python3.12/dist-packages/chainladder/tails/base.py:140: RuntimeWarning: invalid value encountered in multiply
  sigma_ = sigma_ * 0
/usr/local/lib/python3.12/dist-packages/chainladder/tails/base.py:141: RuntimeWarning: invalid value encountered in multiply
  std_err_ = std_err_ * 0
/usr/local/lib/python3.12/dist-packages/chainladder/tails/base.py:133: RuntimeWarning: overflow encountered in exp
  sigma_ = xp.exp(time_pd * reg.slope_ + reg.intercept_)
/usr/local/lib/python3.12/dist-packages/chainladder/tails/base.py:137: RuntimeWarning: overflow encountered in exp
  std_err_ = xp.exp(time_pd * reg.slope_ + reg.intercept_)
/usr/local

/usr/local/lib/python3.12/dist-packages/chainladder/tails/base.py:133: RuntimeWarning: overflow encountered in exp
  sigma_ = xp.exp(time_pd * reg.slope_ + reg.intercept_)
/usr/local/lib/python3.12/dist-packages/chainladder/tails/base.py:137: RuntimeWarning: overflow encountered in exp
  std_err_ = xp.exp(time_pd * reg.slope_ + reg.intercept_)
/usr/local/lib/python3.12/dist-packages/chainladder/tails/base.py:140: RuntimeWarning: invalid value encountered in multiply
  sigma_ = sigma_ * 0
/usr/local/lib/python3.12/dist-packages/chainladder/tails/base.py:141: RuntimeWarning: invalid value encountered in multiply
  std_err_ = std_err_ * 0


/usr/local/lib/python3.12/dist-packages/chainladder/tails/base.py:133: RuntimeWarning: overflow encountered in exp
  sigma_ = xp.exp(time_pd * reg.slope_ + reg.intercept_)
/usr/local/lib/python3.12/dist-packages/chainladder/tails/base.py:137: RuntimeWarning: overflow encountered in exp
  std_err_ = xp.exp(time_pd * reg.slope_ + reg.intercept_)
/usr/local/lib/python3.12/dist-packages/chainladder/tails/base.py:140: RuntimeWarning: invalid value encountered in multiply
  sigma_ = sigma_ * 0
/usr/local/lib/python3.12/dist-packages/chainladder/tails/base.py:141: RuntimeWarning: invalid value encountered in multiply
  std_err_ = std_err_ * 0
/usr/local/lib/python3.12/dist-packages/chainladder/tails/base.py:133: RuntimeWarning: overflow encountered in exp
  sigma_ = xp.exp(time_pd * reg.slope_ + reg.intercept_)
/usr/local/lib/python3.12/dist-packages/chainladder/tails/base.py:137: RuntimeWarning: overflow encountered in exp
  std_err_ = xp.exp(time_pd * reg.slope_ + reg.intercept_)
/usr/local

/usr/local/lib/python3.12/dist-packages/chainladder/tails/base.py:133: RuntimeWarning: overflow encountered in exp
  sigma_ = xp.exp(time_pd * reg.slope_ + reg.intercept_)
/usr/local/lib/python3.12/dist-packages/chainladder/tails/base.py:137: RuntimeWarning: overflow encountered in exp
  std_err_ = xp.exp(time_pd * reg.slope_ + reg.intercept_)
/usr/local/lib/python3.12/dist-packages/chainladder/tails/base.py:140: RuntimeWarning: invalid value encountered in multiply
  sigma_ = sigma_ * 0
/usr/local/lib/python3.12/dist-packages/chainladder/tails/base.py:141: RuntimeWarning: invalid value encountered in multiply
  std_err_ = std_err_ * 0


,volume,simple,regression
claim_count,46.8858,47.5500,46.0641
incurred_amount,"2,220,199.4356","2,564,136.2179","2,000,310.5747"


## 7. Check against the manual calculation

In [12]:
def manual_volume_ibnr(tri_df):
    ldf = [
        tri_df.loc[tri_df[a + 1].notna(), a + 1].sum() / tri_df.loc[tri_df[a + 1].notna(), a].sum()
        for a in range(1, tri_df.shape[1])
    ]
    cdf = np.cumprod(np.append(ldf, 1.0)[::-1])[::-1]
    latest = tri_df.apply(lambda r: r.dropna().iloc[-1], axis=1)
    age = tri_df.apply(lambda r: r.dropna().index[-1], axis=1)
    return (latest * age.map(lambda a: cdf[a - 1]) - latest).sum()


tri_df = count_tri.to_frame(origin_as_datetime=False)
tri_df.columns = range(1, tri_df.shape[1] + 1)
print("package:", round(count_result["ibnr"].sum(), 4))
print("manual :", round(manual_volume_ibnr(tri_df), 4))

package: 46.8858
manual : 46.8858
